## Pull in UDFs

In [1]:
%run nb_udfs

StatementMeta(, ea131ef3-d75d-4fde-aa0f-9e4faf4f8cf6, 13, Finished, Available, Finished, True)

## Run Activities

In [2]:
workspace = 'Fabric of Middle-Earth' #have to escape the & symbol using %26% 
lakehouse = 'lh_the_hoard'
act_table = 'factActivities'
date_offset = 2
#defaults the activity data to grab yesterday's data. The API only goes back 28 days maximum.


StatementMeta(, ea131ef3-d75d-4fde-aa0f-9e4faf4f8cf6, 14, Finished, Available, Finished, False)

In [3]:
"""
Gets the Power BI activity events for a specific day (offset from today).
:param date_offset: Number of days ago (0 = today, 1 = yesterday). Defaults to 1.
"""
# Calculate the UTC date string
target_date = (dt.utcnow() - timedelta(days=date_offset)).strftime('%Y-%m-%d')
start_date = f"{target_date}T00:00:00.00Z"
end_date = f"{target_date}T23:59:59.99Z"
print(f"🔍 Fetching activity for {start_date} to {end_date} - date offset {date_offset}")

StatementMeta(, ea131ef3-d75d-4fde-aa0f-9e4faf4f8cf6, 15, Finished, Available, Finished, False)

🔍 Fetching activity for 2026-11-05T00:00:00.00Z to 2026-11-05T23:59:59.99Z - date offset 2


In [4]:
# Call the API

windows = generate_hourly_windows(target_date)
#print(windows)

#test = list_activity_events('2025-04-19T00:00:00.000Z','2025-04-19T00:59:59.999Z')
#print(test)

# Define dimension load config
activity_path = udf_GetFilePath(workspace, lakehouse, act_table)
natural_key = ['Id']
primary_key = 'tableId'


for start_time, end_time in windows:
    print(f"Fetching events from {start_time} to {end_time}")
    df_activities = list_activity_events(start_time, end_time)
    try:
        df_activities = spark.createDataFrame(df_activities)
        #df_activities.show(1)  # Show a sample if needed
        # Upsert to Delta
        return_val = udf_UpsertDimension(
            df_activities, 
            dimensionType=1,
            targetPath=activity_path,
            naturalKeyColumnList=natural_key,
            primaryKeyColumnName=primary_key,
            flagSoftDeletes=False
        )
        print(return_val)

    except ValueError as e:
        print(f"No activities found for the time window {start_time} to {end_time}. Skipping...")


    





StatementMeta(, ea131ef3-d75d-4fde-aa0f-9e4faf4f8cf6, 16, Submitted, Running, Running, True)

Fetching events from 2026-11-05T00:00:00.000Z to 2026-11-05T00:59:59.999Z
No activities found for the time window 2026-11-05T00:00:00.000Z to 2026-11-05T00:59:59.999Z. Skipping...
Fetching events from 2026-11-05T01:00:00.000Z to 2026-11-05T01:59:59.999Z
No activities found for the time window 2026-11-05T01:00:00.000Z to 2026-11-05T01:59:59.999Z. Skipping...
Fetching events from 2026-11-05T02:00:00.000Z to 2026-11-05T02:59:59.999Z
No activities found for the time window 2026-11-05T02:00:00.000Z to 2026-11-05T02:59:59.999Z. Skipping...
Fetching events from 2026-11-05T03:00:00.000Z to 2026-11-05T03:59:59.999Z
No activities found for the time window 2026-11-05T03:00:00.000Z to 2026-11-05T03:59:59.999Z. Skipping...
Fetching events from 2026-11-05T04:00:00.000Z to 2026-11-05T04:59:59.999Z
No activities found for the time window 2026-11-05T04:00:00.000Z to 2026-11-05T04:59:59.999Z. Skipping...
Fetching events from 2026-11-05T05:00:00.000Z to 2026-11-05T05:59:59.999Z
